# Import the Dataframe

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, AsinhNorm

from scipy.stats import linregress, chi2_contingency, ttest_ind
from sklearn.metrics import confusion_matrix, accuracy_score
from shapely.geometry import Point

from is_in_earth_shadow import *

Read in the cleaned dataframe from the other notebook. 

**Reminder:** Our dataframe has 8352 data points and 32 variables per data point.

In [ ]:
clean_rad = pd.read_excel("cleaned_data.xlsx")

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

geo_rad

Create some subsets of the data based on the 5 major latitude zones so that we can observe trends between the different zones later.

In [ ]:
df_arctic = geo_rad[geo_rad["lat"] > 66.5].copy()
df_antarctic = geo_rad[geo_rad["lat"] < -66.5].copy()
df_tropical = geo_rad[geo_rad["lat"].between(-23.5, 23.5)].copy()
df_stz = geo_rad[geo_rad["lat"].between(-66.5, -23.5, inclusive="neither")].copy()
df_ntz = geo_rad[geo_rad["lat"].between(23.5, 66.5, inclusive="neither")].copy()

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

COLOR = "managua"

# Exploration of the X-Ray Radiation Data

In our meeting on June 18th with Dr. Voss and NearSpace Launch, they directed our focus onto going deeper into the x-ray radiation data. This was the first time that a NearSpace Launch satellite collected x-ray radiation data, so they were unsure how well the collection software was working and general trends among the data.

## Similar Trends Across All 4 X-Ray Detectors

The first step in checking the validity of the x-ray data columns was checking to see if we see similar trends between all 4 of the sensors. These sensors have different thresholds, so if the different different columns show hotspots in different areas than their is likely an issue in the data collection process. 

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    geo_rad['xray0_ps'],
    geo_rad['xray1_ps'],
    geo_rad['xray2_ps'],
    geo_rad['xray3_ps']
])

plt.xticks(
    [1, 2, 3, 4],
    ['xray0','xray1','xray2','xray3']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('X-Ray Radiation Distributions (per second) Across Different Thresholds')
plt.show()

In [ ]:
geo_rad[['xray0_ps', 'xray1_ps', 'xray2_ps', 'xray3_ps']].describe()

This shows us that going forward the higher threshold x-ray radiation data columns are not very beneficial in data analysis. Since the over 75% of all three of the higher threshold columns collected 0 xray radiation waves. This fact may also be hinting at the thresholds being set too high, creating a sensor that can not collect sufficient data since xray radiation waves very rarely reach the required threshold. 

In [ ]:
df_x0 = geo_rad[geo_rad["xray0_ps"] != 0].copy()
df_x1 = geo_rad[geo_rad["xray1_ps"] != 0].copy()
df_x2 = geo_rad[geo_rad["xray2_ps"] != 0].copy()
df_x3 = geo_rad[geo_rad["xray3_ps"] != 0].copy()

When we take out the rows where the sensor registered 0 x-ray radiation in that location, we are left with 
- 7925 data points in 'xray0' (removed 427 data points)
- 1567 data points in 'xray1' (removed 6785 data points)
- 1019 data points in 'xray2' (removed 7333 data points)
- 307 data points in 'xray3' (removed 8045 data points)

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(24, 12))

world.plot(ax=ax1, color="lightgray")
sc1 = ax1.scatter(
    df_x0["lon"],
    df_x0["lat"],
    c=df_x0["xray0_ps"],
    s=5,
    norm=LogNorm()
)
ax1.set_title("X-Ray0 per Second (non-zero values)",fontsize="xx-large")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1, label="X-ray radiation (Log)")

world.plot(ax=ax2, color="lightgray")
sc2 = ax2.scatter(
    df_x1["lon"],
    df_x1["lat"],
    c=df_x1["xray1_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("X-Ray1 per Second (non-zero values)",fontsize="xx-large")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2, label="X-ray radiation (Log)")

world.plot(ax=ax3, color="lightgray")
sc3 = ax3.scatter(
    df_x2["lon"],
    df_x2["lat"],
    c=df_x2["xray2_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray2 per Second (non-zero values)",fontsize="xx-large")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3, label="X-ray radiation (Log)")

world.plot(ax=ax4, color="lightgray")
sc4 = ax4.scatter(
    df_x3["lon"],
    df_x3["lat"],
    c=df_x3["xray3_ps"],
    s=5,
    norm=LogNorm()
)
ax4.set_title("X-Ray3 per Second (non-zero values)",fontsize="xx-large")
ax4.set_xlabel("Longitude")
ax4.set_ylabel("Latitude")
plt.colorbar(sc4, ax=ax4, label="X-ray radiation (Log)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
axes = axes.flatten()

detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    world.plot(ax=axes[i], color="lightgrey")
    norm = AsinhNorm(linear_width=1, vmin=geo_rad[detector].min(), vmax=geo_rad[detector].max())
    geo_rad.plot(
        ax=axes[i],
        markersize=1,
        alpha=0.5,
        column=detector,
        cmap=COLOR,
        norm=norm,
        legend=True
    )
    axes[i].set_title(f"Variable: {detector}")
    axes[i].set_xlabel("Longitude")
    axes[i].set_ylabel("Latitude")

fig.suptitle(f"Xray Detector Counts Per Second (asinh scale)")
plt.show()

### Analysis of asinh detector split
- With the nature of asinh being linear at low values, `xray3_ps` remains entirely unchanged under this scale
- X-rays detected in the Arctic Circle and SAA are especially highlighted in `xray0_ps` and `xray1_ps`
- Disparity between X-ray counts in the northern and southern hemispheres is still visible

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 12), constrained_layout=True)

months = ["Jan", "Feb", "Mar", "Apr"]
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    vmin = geo_rad[detector].min()
    vmax = geo_rad[detector].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for j, month in enumerate(months):
        subset = geo_rad[geo_rad["month"] == month]

        world.plot(ax=axes[i, j], color="lightgrey")
        subset.plot(
            ax=axes[i, j],
            markersize=1,
            alpha=0.5,
            column=detector,
            cmap=COLOR,
            norm=norm,
            legend=True

        )
        axes[i, j].set_title(f"{detector} in {month}")

fig.suptitle(f"X-rays Per Second by Detector and Month",  size='xx-large')
plt.show()

### Analysis of monthly splitting
- Splitting by month shows that all sensors follow similar trends on a monthly basis
- As seen in the previous analyses, X-rays are higheest in the Arctic Circle, though mostly in March and April
- March has the most data, but April has the highest percentage of high values
- The first high values picked up by `xray0_ps` were mostly over Asia and in February
- It might be worth splitting March by week to see if the increase in X-ray values is as correlated with time as it appears when split like this

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 12), constrained_layout=True)

months = ["Jan", "Feb", "Mar", "Apr"]
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    norm = AsinhNorm(linear_width=1, vmin=geo_rad[detector].min(), vmax=geo_rad[detector].max())
    for j, month in enumerate(months):
        subset = geo_rad[geo_rad["month"] == month]

        world.plot(ax=axes[i, j], color="lightgrey")
        subset.plot(
            ax=axes[i, j],
            markersize=1,
            alpha=0.5,
            column=detector,
            cmap=COLOR,
            norm=norm,
            legend=True

        )
        axes[i, j].set_title(f"{detector} in {month}")

fig.suptitle(f"X-rays Per Second by Detector and Month (asinh scale)",  size='xx-large')
plt.show()

### Analysis of monthly splitting scaled by asinh
- X-rays are visibly higher in the SAA in January and February
- With this scale, there are small spikes in X-ray counts in Jaunary and February in the southern hemisphere that were not visible before
- X-rays are incredibly intense in April
- Trends are still holding under this scale

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    geo_rad["lon"],
    geo_rad["lat"],
    c=geo_rad["xray0_ps"],
    s=5
)
ax1.set_title("X-Ray0 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x0["lon"],
    df_x0["lat"],
    c=df_x0["xray0_ps"],
    s=5
)
ax2.set_title("X-Ray0 per Second (non-zero values)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x0["lon"],
    df_x0["lat"],
    c=df_x0["xray0_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray0 per Second (non-zero values) ")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    geo_rad["lon"],
    geo_rad["lat"],
    c=geo_rad["xray1_ps"],
    s=5
)
ax1.set_title("X-Ray1 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x1["lon"],
    df_x1["lat"],
    c=df_x1["xray1_ps"],
    s=5
)
ax2.set_title("X-Ray1 per Second")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x1["lon"],
    df_x1["lat"],
    c=df_x1["xray1_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray1 per Second")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    geo_rad["lon"],
    geo_rad["lat"],
    c=geo_rad["xray2_ps"],
    s=5
)
ax1.set_title("X-Ray2 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x2["lon"],
    df_x2["lat"],
    c=df_x2["xray2_ps"],
    s=5
)
ax2.set_title("X-Ray2 per Second")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x2["lon"],
    df_x2["lat"],
    c=df_x2["xray2_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray2 per Second")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    geo_rad["lon"],
    geo_rad["lat"],
    c=geo_rad["xray3_ps"],
    s=5
)
ax1.set_title("X-Ray3 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x3["lon"],
    df_x3["lat"],
    c=df_x3["xray3_ps"],
    s=5
)
ax2.set_title("X-Ray3 per Second")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x3["lon"],
    df_x3["lat"],
    c=df_x3["xray3_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray3 per Second")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

We can see in the graphs in the first figure that the two major hotspots of the the lowest threshold sensor was the Arctic Circle and the South Atlantic Anomaly. Then when we look at the higher threshold sensors, we see that while most of the data points go away since they recorded 0 radiation. However, most of the locations that the higher threshold sensors did pick up radiation occured in these two areas. Thus, we found that the 4 sensors did have similiar trends and thus did not flag an issue with the sensors functionality. 

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
grouped = clean_rad.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]].mean()

for column in grouped.columns:
    ax.plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

ax.set_title("X-rays Per Second For All Detectors", fontsize="xx-large")
ax.set_xlabel("Group Number")
ax.set_ylabel("Particles Per Second (Log)")

ax.annotate(
    text='Divergence',            # The text display label
    xy=(37, -2.5),                # (x, y) coordinates the arrow points to
    xytext=(-8, -9),              # (x, y) coordinates where the text sits
    arrowprops=dict(
        facecolor='black',        # Arrow fill color
        arrowstyle='->'           # Clean arrow head style
    ),
    fontsize=12,                  # Text sizing
    color='black'                 # Text color
)

ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
axes = axes.flatten()

months = ["Jan", "Feb", "Mar", "Apr"]

for i, month in enumerate(months):
    subset = clean_rad[clean_rad["month"] == month]
    grouped = subset.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Month: {month}", fontsize="xx-large")
    axes[i].set_ylim(-10, 10)

fig.suptitle("Group averages for all xray detectors by month", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper left", fontsize="x-large")
plt.show()

### Analysis of linegraphs
- Just like we've already seen with previous analysis, trends are almost always holding between detectors
- There is some potential noise here:
    - To get the most accurate graph, I would need to plot the x-axis by timestamp
        - Plotting with timestamp makes for results that are very hard to interpret since observations can be 4 seconds apart or 18 hours apart
    - I grouped them by `group` and plotted each group's mean instead
        - This data is full of outliers, but mean has to do.
            - Median and mode both graph 0 for almost all of `xray2_ps` and `xray3_ps`
        - Grouping and plotting by mean may allow for outliers to skew data, but we can see similar trends emerge in each detector anyway

### Get ratio of all detector combinations

In [ ]:
detectors = ["xray0", "xray1", "xray2", "xray3"]

fig, axes = plt.subplots(4, 4, figsize=(20, 20), constrained_layout=True)

grouped = clean_rad.groupby("group")[["xray0", "xray1", "xray2", "xray3"]].mean()

for i, detector in enumerate(detectors):
    for j, detector2 in enumerate(detectors):
        axes[i, j].plot(grouped.index, grouped[detector]/grouped[detector2])
        axes[i, j].set_yscale('log')
        axes[i, j].set_title(f"{detector}/{detector2}")
        axes[i, j].set_ylabel("Particle Counts Per Second (Log)")
        axes[i, j].set_xlabel("Group")

fig.suptitle("X-ray Detector Ratios", fontsize="xx-large")
plt.show()

### Analysis of detector ratios
One last test to show similar trends

- It appears that each X-ray sensor has an energy threshold 10 times higher than the previous detector
- As you move down the visual, the graphs stay a similar shape
- As you move from left to right on the visual, graphs stay a similar shape but develop more holes
    - These holes develop because dividing by the `xray2_ps` and `xray3_ps` columns start to involve a lot of divisions by 0.
- These shapes staying the same between ratios shows yet again that there are similar trends between the four X-ray detectors

## X-Ray and Electron Radiation

One potential issue that Dr. Voss and NearSpace Launch raised on the functionality of their x-ray radiation sensors was the possibility that electron radiation was leaking into the x-ray sensor and being counted as x-ray radiation. Thus looking into that possiblity through relationship between the x-ray radiation and electron radiation columns.

### Linear Regression t-Tests

In a linear regression t-test, our hypotheses are...

- $H_0: \beta = 0$
- $H_a: \beta \neq 0$

then we will reject the null hypothesis if our p-value is less than 0.05

In [ ]:
def lin_reg_ttest(x,y):
    for i in x:
        for j in y:
            result = linregress(geo_rad[i],geo_rad[j])
            slope = result.slope
            intercept = result.intercept
            r_squared = result.rvalue**2
            p_value = result.pvalue
            t_stat = result.slope / result.stderr
            
            y_fit = slope * geo_rad[i] + intercept
            
            plt.figure(figsize=(8, 6))
            plt.scatter(geo_rad[i], geo_rad[j], s=5, alpha=0.5)
            plt.plot(geo_rad[i], y_fit, color="red")
            
            plt.xlabel(i)
            plt.ylabel(j)
            plt.title(i + ' vs ' + j)
            
            plt.show()
            
            print(f"Slope: {slope:.6f}")
            print(f"Intercept: {intercept:.6f}")
            print(f"R²: {r_squared:.4f}")
            print(f"t-statistic: {t_stat:.4f}")
            print(f"p-value: {p_value:.6g}")

In [ ]:
x = geo_rad["xray0"]
y = geo_rad["electron0"]

result = linregress(x, y)

slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2
p_value = result.pvalue
t_stat = result.slope / result.stderr

y_fit = slope * x + intercept

plt.figure(figsize=(8, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x, y_fit, color="red")

plt.xlabel("xray0")
plt.ylabel("electron0")
plt.title("xray0 vs electron0")

plt.show()

print(f"Slope: {slope:.6f}")
print(f"Intercept: {intercept:.6f}")
print(f"R²: {r_squared:.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6g}")

**Linear Regression t-Test between 'xray0' and 'electron0'**

We are running a linear regression t-test to see if there is a relationship between the 'xray0' and 'electron0' columns. Since the test yields a p-value that is extremely small, we will reject the null hypothesis and conclude that the slope is statistically significant. However, the statistically signifant slope is very gradual and inverted (-0.03). This means that for every additional x-ray wave picked up by our sensor, our electron radiation sensor should pick up 0.03 less electrons. Another thing to note is that our $R^2$ value is also extremely small (0.0099), which means that the linear regression line only accounts for 0.99% of the variation in the electron radiation. 

Some things to note from this test are...
1. We expected to see a positive relationship not an inverse one, which makes the result very interesting
2. While statistically significant the slope of the regression line accounts for a very small percentage of the variation in electron radiation

In [ ]:
xlab = ['xray0','xray1','xray2','xray3']
ylab = ['electron0','electron1']

lin_reg_ttest(xlab, ylab)

As we moved past the lowest threshold x-ray sensor, we started to see the positive correlation between the two sensors that Dr. Voss and NearSpace Launch was worried about. Also, all 8 combinations of the linear regression t-tests yielded a low p-value and thus a statistically significant slope. However, the greatest $R^2$ value was just over 0.03, and thus none of the linear regression lines accounted for a large portion of the variation in the electron sensors. Therefore, we can't draw any strong conclusions from these tests as to if the electron radiation was leaking into the x-ray sensors and skewing the data. If this is something that NearSpace Launch finds concerning, then they should try to run some additional tests and see if the electron radiation leakage is really occuring in their x-ray sensors.

### Mix of X-Ray and Electron Radiation in X-Ray Sensors

In [ ]:
geo_rad["electron0 + xray1"] = geo_rad["electron0"] + geo_rad["xray1"]
geo_rad["electron0 + xray2"] = geo_rad["electron0"] + geo_rad["xray2"]
geo_rad["electron0 + xray3"] = geo_rad["electron0"] + geo_rad["xray3"]
geo_rad["electron1 + xray1"] = geo_rad["electron1"] + geo_rad["xray1"]
geo_rad["electron1 + xray2"] = geo_rad["electron1"] + geo_rad["xray2"]
geo_rad["electron1 + xray3"] = geo_rad["electron1"] + geo_rad["xray3"]

In [ ]:
xlabs = ['xray0','xray1','xray2','xray3']
ylabs = ['electron0 + xray1','electron0 + xray2','electron0 + xray3','electron1 + xray1','electron1 + xray2','electron1 + xray3']

lin_reg_ttest(xlabs,ylabs)

All of the linear regression t-tests between the x-ray radiation columns and the new columns the was the sum of higher threshold x-ray and/or electron radiation columns gave similar results as the simple x-ray vs electron radiation columns. That is that the slope is statistically significance due to low p-values, but also does not account for much of the variation in the y-axis sums due to the low $R^2$ values. 

## X-Ray During Day vs Night 

### Check Validity of Column Values

NearSpace Launch informed us that were uncertain as to the validity of the "in_shadow" column of their data set. I was able to find a library that allowed us to make a function that uses the timestamp, latitude, longitude, and altitude data of the satellite to check if the Earth was in between the sun and our satellite.

In [ ]:
geo_rad["shadow_check"] = geo_rad.apply(
    lambda row: is_in_earth_shadow(
        row["lat"],
        row["lon"],
        row["alt"],
        row["timestamp"]
    ),
    axis=1
)

In [ ]:
geo_rad["shadow_check"] = geo_rad["shadow_check"].astype(int)
geo_rad = geo_rad.dropna(subset=["in_shadow"])

In [ ]:
cm = confusion_matrix(geo_rad["shadow_check"],geo_rad["in_shadow"],labels=[0, 1])
print(cm)

accuracy = accuracy_score(geo_rad["shadow_check"],geo_rad["in_shadow"])
print(f"Accuracy: {accuracy:.4f}")

When we run our function on our data and then compare the function's in shaodw value with the original in shadow values, we can get that 99.96% of the values are the same. In fact, there were only 3 values that differed between the function and the original, and all 3 of these values were deemed not in shadow in the original column but and deemed in shadow using the function. We decided to just use the original in shadow values going forward since the similarity was so similar between the two columns. 

### Exploratory

In [ ]:
day = geo_rad[geo_rad['in_shadow']==0].copy()
night = geo_rad[geo_rad['in_shadow']==1].copy()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

world.plot(ax=ax1, color="lightgray")
ax1.scatter(
    day["lon"],
    day["lat"],
    c='black',
    s=5
)
ax1.set_title("Daytime Locations")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")

world.plot(ax=ax2, color="lightgray")
ax2.scatter(
    night["lon"],
    night["lat"],
    c='black',
    s=5
)
ax2.set_title("Nighttime Locations")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

These two graphs show us the locations in which data was registered when the satellite was both in the Earth's shadow and when it was not. One concerning thing that we noticed from these graphs is the lack of in shadow data from both the Arctic and Antarctic Circles. 

In [ ]:
plt.figure(figsize=(8, 6))

plt.boxplot([
    day['xray0_ps'],
    night['xray0_ps']
])

plt.xticks(
    [1, 2],
    ['In Sun','In Shadow']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('X-Ray Radiation Distributions (per second)')
plt.show()

In [ ]:
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps", "ses_ps", "total_radiation_ps"]

for detector in detectors:
    subset = {
        "north, day": clean_rad[(clean_rad["lat"] >= 0) & (clean_rad["in_shadow"] == 0)][detector],
        "north, night": clean_rad[(clean_rad["lat"] >= 0) & (clean_rad["in_shadow"] == 1)][detector],
        "south, day": clean_rad[(clean_rad["lat"] < 0)  & (clean_rad["in_shadow"] == 0)][detector],
        "south, night": clean_rad[(clean_rad["lat"] < 0)  & (clean_rad["in_shadow"] == 1)][detector],
    }

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.boxplot(
        subset.values(),
        tick_labels=subset.keys(),
        patch_artist=True,
        boxprops=dict(facecolor="steelblue", alpha=0.6),
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker="o", markersize=2, alpha=0.3)
    )

    ax.set_title(f"{detector} by hemisphere and in_shadow", fontsize="xx-large")
    ax.set_xlabel(f"Hemisphere and Illumination")
    ax.set_ylabel(f"{detector} (log)")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()

#### Analysis of boxplots
As we've already seen, we observe more X-rays in the northern hemisphere than the souther. We see more protons and electrons in the southern hemisphere. `ses_ps` appears to follow a very similar trend to the protons and electrons in all hemisphere and `in_shadow` combinations. Our total radiation appears to have a higher median in the sunlight than in shadow

In [ ]:
print('IN SUN')
print(day['xray0_ps'].describe())

print()
print('IN SHADOW')
print(night['xray0_ps'].describe())

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    day['xray0_ps'],
    night['xray0_ps'],
    day['electron0_ps'],
    night['electron0_ps'],
    day['proton0_ps'],
    night['proton0_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5, 6],
    ['In Sun X-Ray','In Shadow X-Ray','In Sun Electron','In Shadow Electron','In Sun Proton','In Shadow Proton']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('Electron and Proton Radiation Distributions (per second)')
plt.show()

In [ ]:
print('ELECTRON IN SUN')
print(day['electron0_ps'].describe())

print()
print('ELECTRON IN SHADOW')
print(night['electron0_ps'].describe())

In [ ]:
print('PROTON IN SUN')
print(day['proton0_ps'].describe())

print()
print('PROTON IN SHADOW')
print(night['proton0_ps'].describe())

This shows us that there is a large difference in the all three types of radiation when our satellite can see the sun versus when the satellite is in the Earth's shadow, with all three types of radiation being higher when the satellite is in the sun. However, we will need to do more data analysis on this in order to prove that this difference is not due to some confounding variables. For example, in the first visual in this secton we saw the locations of the in sun versus in shadow data points, and in that visual we noted that all of the in shadow data points fell in the middle more tropical regions of the earth with very few in shadow data points in the Arctic and Antarctic Circles. 

#### Welsh's Two-Sample t-Tests

In Welch's two-sample t-test, our hypotheses are...

- $H_0: \bar{x}_{day} = \bar{x}_{night}$
- $H_a: \bar{x}_{day} \neq \bar{x}_{night}$

then we will reject the null hypothesis if our p-value is less than 0.05.

In [ ]:
t_stat, p_value = ttest_ind(day['xray0_ps'], night['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

First we take the full dataset and look at the x-ray radiation distributions between when the satellite is in direct sunlight and when it is in the Earth's shadow. The Welsh's Two-Sample t_Test on the resulting distribution yields a p-value that is virtually 0, thus we reject the null hypothesis and conclude that the mean amount of x-ray radiation in the sun is different then the mean amount of radiation in the Earth's shadow. 

In [ ]:
t_stat, p_value = ttest_ind(day['electron0_ps'], night['electron0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
t_stat, p_value = ttest_ind(day['proton0_ps'], night['proton0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
geo_rad["region"] = np.select(
    [
        geo_rad["lat"] > 66.5,
        geo_rad["lat"] < -66.5
    ],
    [
        "Arctic",
        "Antarctic"
    ],
    default="Tropical"
)

In [ ]:
table = pd.crosstab(
    geo_rad["region"],
    geo_rad["in_shadow"]
)

print(table)

In [ ]:
day_tropical = day[day["lat"].between(-66.5, 66.5)].copy()
night_tropical = night[night["lat"].between(-66.5, 66.5)].copy()
day_arctic = day[day["lat"] > 66.5].copy()
night_arctic = night[night["lat"] > 66.5].copy()
day_antarctic = day[day["lat"] < -66.5].copy()
night_antarctic = night[night["lat"] < -66.5].copy()

In [ ]:
t_stat, p_value = ttest_ind(day_tropical['xray0_ps'], night_tropical['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

I believe that this central region of the globe will give us the most accurate test as there is a substantial amount of data for both the day and night distributions. In this Welsh Two-Sample t-Test, we also get a p-value that is virtually 0 and thus will again reject the null hypothesis in favor of there being differing means between when the satellite is in the sun versus in the Earth's shadow. 

In [ ]:
t_stat, p_value = ttest_ind(day_arctic['xray0_ps'], night_arctic['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
t_stat, p_value = ttest_ind(day_antarctic['xray0_ps'], night_antarctic['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

While the tests might be slightly inaccurate in the Arctic and Antarctic Circles due to the lack of in shadow data points, I figured that I would run the tests anyways. Once again for both the Arctic and Antarctic Circles, we get p-values that is virtually 0 and thus will again reject the null hypothesis in favor of there being differing means between when the satellite is in the sun versus in the Earth's shadow. 

In [ ]:
x = day["lat"]
y = day["xray0_ps"]

result = linregress(x, y)

slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2

y_fit = slope * x + intercept

plt.figure(figsize=(12, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x, y_fit)

plt.xlabel("Latitude")
plt.ylabel("X-Ray 0")
plt.title("X-Ray vs Latitude (Daytime)")

plt.show()

print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R²: {r_squared:.4f}")

In [ ]:
x = night["lat"]
y = night["xray0_ps"]

result = linregress(x, y)

slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2

y_fit = slope * x + intercept

plt.figure(figsize=(12, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x, y_fit)

plt.xlabel("Latitude")
plt.ylabel("X-Ray 0")
plt.title("X-Ray vs Latitude (Nighttime)")

plt.show()

print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R²: {r_squared:.4f}")

In [ ]:
x = day["lat"]
y = day["xray0_ps"]

# Fit quadratic: y = ax² + bx + c
a, b, c = np.polyfit(x, y, 2)

# Create smooth curve for plotting
x_fit = np.linspace(x.min(), x.max(), 1000)
y_fit = a * x_fit**2 + b * x_fit + c

# Calculate R²
y_pred = a * x**2 + b * x + c

ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - np.mean(y))**2)
r_squared = 1 - ss_res / ss_tot

# Plot
plt.figure(figsize=(12, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x_fit, y_fit, linewidth=2)

plt.xlabel("Latitude")
plt.ylabel("X-Ray 0")
plt.title("X-Ray vs Latitude (Nighttime)")

plt.show()

print(f"y = {a:.6e}x² + {b:.6e}x + {c:.6e}")
print(f"R² = {r_squared:.4f}")

In [ ]:
arctic_day = df_arctic[df_arctic["in_shadow"] == 0]
arctic_night = df_arctic[df_arctic["in_shadow"] == 1]
antarctic_day = df_antarctic[df_antarctic["in_shadow"] == 0]

data = [arctic_day, arctic_night, antarctic_day]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

colors = ["blue", "orange", "green", "red", "purple", "pink"]

for i, df in enumerate(data):
    grouped = df.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]].mean()

    for j, column in enumerate(grouped.columns):
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second", color=colors[j])

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"}")
    axes[i].set_ylabel("Particles Per Second (Log)")

fig.suptitle("Group averages for all sensors", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="medium")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), constrained_layout=True)

for i, df in enumerate(data):
    grouped = df.groupby("group")[["xray0_ps", "proton0_ps", "electron0_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"}", fontsize="x-large")
    axes[i].set_ylabel("Particles Per Second (Log)")
    axes[i].set_xlabel("Group Number")


fig.suptitle("Group averages for all sensors", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="medium")
plt.show()

### Analysis of line graphs
I was not expecting `xray0_ps` to follow the trends of `proton0_ps` and `electron0_ps` in the Antarctic Circle in sunlight. That is fascinating and absolutely something we should look into further. `xray0_ps` follows the trends of the other two in the Arctic circle as well, but it deviates more often in those graphs. 
 
The large gap in data in the Antarctic Circle is very evident here.

It looks like `proton0_ps` and `electron0_ps` dip way lower than `xray0_ps` and almost rhythmically in the Arctic Circle in the latter half of the data collection period. That's around when the X-rays become more prevalent in the Northern hemisphere. 

The other X-ray sensors still follow the trends of `xray0_ps`, though the values are all very, very low in comparison. This is consistent with what we've already seen.
 
It's interesting how high all of the values are in January

### Plot the same data in a box and whisker plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes = axes.flatten()

for i, df in enumerate(data):

    subset = df[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]]

    bp = axes[i].boxplot(subset,
                         tick_labels=subset.keys(),
                         patch_artist=True,
                         boxprops=dict(facecolor="steelblue", alpha=0.6),
                         medianprops=dict(color="black", linewidth=2),
                         flierprops=dict(marker="o", markersize=2, alpha=0.3)
    )

    for median_line in bp['medians']:
        median_val = median_line.get_ydata()[0]
        left_edge_x = median_line.get_xdata()[0]

        axes[i].hlines(y=median_val, xmin=axes[i].get_xlim()[0], xmax=left_edge_x, color='gray', linestyle='--', linewidth=1)
    
    axes[i].set_yscale("log")
    axes[i].set_ylim(10**-3, 10**5)
    axes[i].set_xlabel("Radiation Sensor")
    axes[i].set_ylabel("Particles Per Second (Log)")
    axes[i].set_title(f"Radiation in {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} (Data points: {len(df)})")

fig.delaxes(axes[3])
plt.suptitle("All sensors per second by pole and daylight", fontsize="xx-large")
plt.tight_layout()
plt.show()

### Analysis of box and whisker plot

This could be a sample size difference, but X-rays defintely appear to be most prevalent in the sunlight and in the northern hempiphere. It already appeared that way in previous analysis, but it's visible here as well.

Interestingly, `proton0_ps` are not super far apart between the Arctic and Antarctic circles in the daylight. 

Electrons are clearly higher in the southern hemisphere. They are also steady in the Arctic circle regardless of daylight.

## How does the radiation differ when it's dark on the ground?

### Create a column similar to `in_shadow` that reports if it is in shadow at sea level

In [ ]:
clean_rad['alt0'] = 0

clean_rad["ground_shadow"] = clean_rad.apply(
    lambda row: is_in_earth_shadow(
        row["lat"],
        row["lon"],
        row["alt0"],
        row["timestamp"]
    ),
    axis=1
).replace({True: 1, False: 0})

clean_rad.drop("alt0", axis=1, inplace=True)

In [ ]:
for circle in [(arctic, "Arctic"), (antarctic, "Antarctic")]:
    for shadow in [0, 1]:
        print(f"Data points in {circle[1]} Circle when in {"shadow" if shadow else "sunlight"}: {len(circle[0][circle[0]["in_shadow"] == shadow])}")

### Check the sizes of all combinations of `ground_shadow`, `in_shadow`, and the poles

In [ ]:
print(f"Size of Arctic satellite day:\t\t {len(clean_rad[(clean_rad["in_shadow"] == 0) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Arctic ground day:\t\t {len(clean_rad[(clean_rad["ground_shadow"] == 0) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Arctic satellite shadow:\t {len(clean_rad[(clean_rad["in_shadow"] == 1) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Arctic ground shadow:\t\t {len(clean_rad[(clean_rad["ground_shadow"] == 1) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Antarctic satellite shadow:\t {len(clean_rad[(clean_rad["in_shadow"] == 0) & (clean_rad["lat"] <= -66.5)])}")
print(f"Size of Antarctic ground shadow:\t {len(clean_rad[(clean_rad["ground_shadow"] == 0) & (clean_rad["lat"] <= -66.5)])}")
print(f"Size of Antarctic satellite shadow:\t {len(clean_rad[(clean_rad["in_shadow"] == 1) & (clean_rad["lat"] <= -66.5)])}")
print(f"Size of Antarctic ground shadow:\t {len(clean_rad[(clean_rad["ground_shadow"] == 1) & (clean_rad["lat"] <= -66.5)])}")


### Redefine our test dataframes now that we have a new column

In [ ]:
arctic = clean_rad[clean_rad["lat"] >= 66.5]
antarctic = clean_rad[clean_rad["lat"] <= -66.5]

ground_arctic_day = arctic[arctic["ground_shadow"] == 0]
ground_arctic_night = arctic[arctic["ground_shadow"] == 1]
ground_antarctic_day = antarctic[antarctic["ground_shadow"] == 0]
ground_antarctic_night = antarctic[antarctic["ground_shadow"] == 1]

ground_data = [ground_arctic_day, ground_arctic_night, ground_antarctic_day, ground_antarctic_night]

### Compare group averages on the ground

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

colors = ["blue", "orange", "green", "red", "purple", "pink"]

for i, df in enumerate(ground_data):
    grouped = df.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]].mean()

    for j, column in enumerate(grouped.columns):
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second", color=colors[j])

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} on the ground")
    axes[i].set_ylabel("Particles Per Second (Log)")

fig.suptitle("Group averages for all sensors for in_shadow on the ground", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="large")
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 8), constrained_layout=True)

for i, df in enumerate(ground_data):
    grouped = df.groupby("group")[["xray0_ps", "proton0_ps", "electron0_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} on the ground", fontsize="x-large")
    axes[i].set_ylabel("Particles Per Second (Log)")
    axes[i].set_xlabel("Group Number")



fig.suptitle("Group averages for main sensors for in_shadow on the ground", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", bbox_to_anchor=(1.002, 1.02), fontsize="medium")
plt.show()

### Compare sensors on the ground with a box and whisker plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes = axes.flatten()

for i, df in enumerate(ground_data):

    subset = df[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]]

    bp = axes[i].boxplot(subset,
                         tick_labels=subset.keys(),
                         patch_artist=True,
                         boxprops=dict(facecolor="steelblue", alpha=0.6),
                         medianprops=dict(color="black", linewidth=2),
                         flierprops=dict(marker="o", markersize=2, alpha=0.3)
    )

    for median_line in bp['medians']:
        median_val = median_line.get_ydata()[0]
        left_edge_x = median_line.get_xdata()[0]

        axes[i].hlines(y=median_val, xmin=axes[i].get_xlim()[0], xmax=left_edge_x, color='gray', linestyle='--', linewidth=1)
    
    axes[i].set_yscale("log")
    axes[i].set_ylim(10**-3, 10**5)
    axes[i].set_ylabel("Particles Per Second (Log)")
    axes[i].set_title(f"Radiation in {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} (Data points: {len(df)})")

# fig.delaxes(axes[3])
plt.suptitle("All sensors per second by pole and daylight on the ground", fontsize="xx-large")
plt.tight_layout()
plt.show()

### Analysis of box and whisker plot on the ground

It appears that a lot of the low values for `xray0_ps` where `in_shadow == 0` fall under `ground_shadow == 1`. This could be a coincidence, but it looks like sunlight hitting the surface may have a potential effect on X-ray readings. This could be due to X-rays reflecting back up from the surface, or, as mentioned, it could be complete chance. More analysis would be required to know for sure. Barring that, most things stayed fairly similar between box plots.

### Compare ground daylight to satellite daylight

In [ ]:
all_data = [(arctic_day, ground_arctic_day), 
            (arctic_night, ground_arctic_night), 
            (antarctic_day, ground_antarctic_day), 
            (ground_antarctic_night, ground_antarctic_night)]

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 12))

axes = axes.flatten()

for i, dfs in enumerate(all_data):
    for j in range(2):
        subset = dfs[j][["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]]

        bp = axes[2*i+j].boxplot(subset,
                            tick_labels=subset.keys(),
                            patch_artist=True,
                            boxprops=dict(facecolor="steelblue", alpha=0.6),
                            medianprops=dict(color="black", linewidth=2),
                            flierprops=dict(marker="o", markersize=2, alpha=0.3)
        )

        for median_line in bp['medians']:
            median_val = median_line.get_ydata()[0]
            left_edge_x = median_line.get_xdata()[0]

            axes[2*i+j].hlines(y=median_val, xmin=axes[i].get_xlim()[0], xmax=left_edge_x, color='gray', linestyle='--', linewidth=1)

        
        axes[2*i+j].set_yscale("log")
        axes[2*i+j].set_ylim(10**-3, 10**5)
        axes[2*i+j].set_title(f"Radiation in {"Arctic" if i < 2 else "Antarctic"} Circle when {"satellite" if (j % 2 == 0) else "ground"} in {"shadow" if (i % 2 != 0) else "sunlight"} (Data points: {len(dfs[j])})", fontsize="medium")

axes[6].set_visible(False)
plt.suptitle("All sensors per second by pole and daylight on the ground", fontsize="xx-large")
plt.tight_layout()
plt.show()

### Analysis of ground daylight vs satellite daylight

This is an interesting graph, but I'm not sure how helpful it really is. Sample size issues make it fairly hard to compare graphs to each other, we don't have enough points to analyze the Antarctic Circle with the satellite in shadow with any integrity, and there is significant overlap in the `in_shadow` and `ground_shadow` columns. I created it hoping to see some sort of dramatic difference, but there isn't one. 
  
`xray0_ps` shows an order of magnitude difference between satellite sunlight and ground sunlight (and the same for the lack thereof), but these, again, could come from diffrences in sample sizes and overlaps between the data. Ground sunlight is effectively just a smaller version of satellite sunlight here. Same with ground shadow and satellite shadow. With those issues and `xray0_ps` being the only column showing any differences, I'm concerned about drawing any conclusions here. Maybe the experts will have something else to say about it, though.